# Notebook 08 — Structural Features Deep Dive

This notebook analyses the 17 structural features used by the anomaly detection pipeline.  Unlike TF-IDF or LSA representations, structural features are computed directly from raw text — no vocabulary, no dimensionality reduction, no NLP pipeline.  They capture surface patterns such as lexical diversity, repetition, and information density.

Two of these features were added because they measure lexical and informational diversity in a way that reliably separates highly repetitive documents from natural prose:

- **Bigram type-token ratio** — the fraction of bigrams that are unique.  Templated text reuses the same phrases repeatedly, driving this value close to 0.  Natural text tends to be near 1.
- **Compression ratio** — the fraction `compressed_bytes / raw_bytes` using gzip.  Repetitive documents compress to a much smaller fraction of their original size than diverse natural prose.

The notebook covers:
1. Computing all 17 structural features for the full corpus.
2. Summary statistics comparing low-scoring and high-scoring documents.
3. Per-feature distributions with 5th-percentile markers.
4. Deep-dive distributions for bigram TTR and compression ratio with text examples.
5. Joint scatter plot of the two new features coloured by Isolation Forest anomaly score.
6. Feature correlation heatmap.
7. The 5 most extreme documents for each feature with text snippets.

## Setup

Add the `src/` directory to the path so that the project modules are importable without installing the package.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest

from core.data_io import ArticleDataset
from core.paths import PipelinePaths
from preprocessing import StructuralFeatureExtractor

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 4)
sns.set_style("whitegrid")

RESULTS_DIR = project_root / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Results dir  : {RESULTS_DIR}")

## Load corpus and compute structural features

The `StructuralFeatureExtractor` works on raw text — it is called before any normalisation step.  Every document in the 2 164-document corpus is processed to produce a `(2164, 17)` feature matrix.

In [ ]:
pipeline_paths = PipelinePaths.from_project_root(project_root)
dataset = ArticleDataset(input_csv_path=pipeline_paths.input_articles_csv)
articles_df = dataset.load_articles()

extractor = StructuralFeatureExtractor()
feature_names = extractor.get_feature_names()
feature_matrix = extractor.transform(articles_df["text"].astype(str).tolist())

print(f"Corpus size      : {len(articles_df):,} documents")
print(f"Feature matrix   : {feature_matrix.shape}")
print(f"Features ({len(feature_names)}): {feature_names}")

## Summary statistics across all features

The table below shows descriptive statistics (mean, std, min, 5th and 95th percentiles, max) for each structural feature.  Features with a wide spread between the 5th and 95th percentiles are more discriminative.

In [ ]:
feature_df = pd.DataFrame(feature_matrix, columns=feature_names)
feature_df.insert(0, "doc_id", articles_df["doc_id"].values)

stats = feature_df[feature_names].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
stats.columns = ["count", "mean", "std", "min", "p5", "p25", "median", "p75", "p95", "max"]
stats = stats.drop(columns="count")
pd.set_option("display.float_format", "{:.4f}".format)
stats

## Per-feature distribution overview

All 17 features are plotted as histograms in a 6×3 grid.  A vertical dashed line marks the 5th percentile — documents to the left of that line are in the most anomalous tail for that feature.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(14, 18))
axes_flat = axes.flatten()

for idx, feature_name in enumerate(feature_names):
    ax = axes_flat[idx]
    values = feature_df[feature_name].values
    p5 = np.percentile(values, 5)
    ax.hist(values, bins=50, color="steelblue", edgecolor="none", alpha=0.8)
    ax.axvline(p5, color="crimson", linestyle="--", linewidth=1.2, label=f"p5={p5:.3f}")
    ax.set_title(feature_name, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("count", fontsize=8)
    ax.legend(fontsize=7, frameon=False)
    ax.tick_params(labelsize=7)

# Hide the last empty subplot if feature count < 18.
for idx in range(len(feature_names), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Structural feature distributions (5th-percentile marker in red)", fontsize=12, y=1.01)
plt.tight_layout()
overview_path = RESULTS_DIR / "structural_features_overview.png"
plt.savefig(overview_path, bbox_inches="tight")
plt.show()
print(f"Saved → {overview_path}")

## Deep dive: Bigram type-token ratio

### What it measures

The bigram type-token ratio (bigram TTR) is defined as:

$$\text{bigram TTR} = \frac{|\{\text{unique bigrams}\}|}{|\text{bigrams}|}$$

A value of **1.0** means every consecutive pair of words in the document is a different phrase — highly creative, diverse text.  A value close to **0** means the same phrases are repeated over and over again, which is characteristic of templated content.

### Why low bigram TTR signals anomalous documents

Documents flagged as anomalous by the detector tend to score very low on this metric.  Inspection of the lowest-scoring documents (see code cells below) reveals heavily templated content with repeated phrasing.  Normal newsgroup posts achieve values close to 1.0 because each sentence builds on different vocabulary.

In [ ]:
bttr_values = feature_df["bigram_type_token_ratio"].values
p5_bttr = np.percentile(bttr_values, 5)
low_bttr_mask = bttr_values <= p5_bttr

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full distribution.
ax = axes[0]
ax.hist(bttr_values, bins=60, color="steelblue", edgecolor="none", alpha=0.85)
ax.axvline(p5_bttr, color="crimson", linestyle="--", linewidth=1.5, label=f"p5 = {p5_bttr:.3f}")
ax.set_title("Bigram TTR — full corpus")
ax.set_xlabel("bigram type-token ratio")
ax.set_ylabel("document count")
ax.legend(frameon=False)

# Zoomed tail.
ax2 = axes[1]
tail_values = bttr_values[bttr_values < 0.5]
ax2.hist(tail_values, bins=40, color="salmon", edgecolor="none", alpha=0.85)
ax2.axvline(p5_bttr, color="crimson", linestyle="--", linewidth=1.5, label=f"p5 = {p5_bttr:.3f}")
ax2.set_title("Bigram TTR — low-value tail (< 0.5)")
ax2.set_xlabel("bigram type-token ratio")
ax2.set_ylabel("document count")
ax2.legend(frameon=False)

plt.suptitle("Bigram Type-Token Ratio distribution", fontsize=12)
plt.tight_layout()
bttr_path = RESULTS_DIR / "structural_bttr_distribution.png"
plt.savefig(bttr_path, bbox_inches="tight")
plt.show()

print(f"Documents with bigram TTR ≤ p5 ({p5_bttr:.3f}) : {low_bttr_mask.sum():,}")
print(f"Corpus median bigram TTR                        : {np.median(bttr_values):.4f}")
print(f"Saved → {bttr_path}")

### Example documents with very low bigram TTR

The five documents with the lowest bigram TTR show the most intense phrase repetition in the corpus.

In [ ]:
bttr_series = feature_df.set_index("doc_id")["bigram_type_token_ratio"]
text_series = articles_df.set_index("doc_id")["text"]

print("=" * 80)
print("TOP 5 LOWEST BIGRAM TTR DOCUMENTS")
print("=" * 80)
for doc_id in bttr_series.nsmallest(5).index:
    score = bttr_series[doc_id]
    snippet = str(text_series[doc_id])[:400].replace("\n", " ")
    print(f"\ndoc_id={doc_id}  bigram_TTR={score:.4f}")
    print("-" * 60)
    print(snippet + " ...")

## Deep dive: Compression ratio

### What it measures

The compression ratio is:

$$\text{compression ratio} = \frac{|\text{gzip}(\text{text})|}{|\text{UTF-8 bytes of text}|}$$

gzip (DEFLATE) exploits repeated byte sequences to shrink data.  A value of **1.0** would mean the compressed file is as large as the original — the text is maximally incompressible, i.e. every bit is unpredictable.  A very **low** value means the document is dominated by repeated patterns that compress away almost entirely.

### Connection to Kolmogorov complexity

The compression ratio is a computable proxy for Kolmogorov complexity — the length of the shortest program that outputs the string.  A short program can generate highly repetitive text with a simple loop.  Natural language, by contrast, requires a longer program because each sentence introduces novel structure.

Documents in the low-compression tail of the corpus show a striking gap from the main distribution, visible in the histograms below.  The exact medians for flagged vs normal documents are computed from the Isolation Forest results later in this notebook.

In [ ]:
cr_values = feature_df["compression_ratio"].values
p5_cr = np.percentile(cr_values, 5)
low_cr_mask = cr_values <= p5_cr

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(cr_values, bins=60, color="mediumseagreen", edgecolor="none", alpha=0.85)
ax.axvline(p5_cr, color="crimson", linestyle="--", linewidth=1.5, label=f"p5 = {p5_cr:.3f}")
ax.set_title("Compression ratio — full corpus")
ax.set_xlabel("gzip compression ratio")
ax.set_ylabel("document count")
ax.legend(frameon=False)

ax2 = axes[1]
tail_cr = cr_values[cr_values < 0.55]
ax2.hist(tail_cr, bins=40, color="lightcoral", edgecolor="none", alpha=0.85)
ax2.axvline(p5_cr, color="crimson", linestyle="--", linewidth=1.5, label=f"p5 = {p5_cr:.3f}")
ax2.set_title("Compression ratio — low-value tail (< 0.55)")
ax2.set_xlabel("gzip compression ratio")
ax2.set_ylabel("document count")
ax2.legend(frameon=False)

plt.suptitle("Compression ratio distribution", fontsize=12)
plt.tight_layout()
cr_path = RESULTS_DIR / "structural_compression_ratio_distribution.png"
plt.savefig(cr_path, bbox_inches="tight")
plt.show()

print(f"Documents with compression ratio ≤ p5 ({p5_cr:.3f}) : {low_cr_mask.sum():,}")
print(f"Corpus median compression ratio                     : {np.median(cr_values):.4f}")
print(f"Saved → {cr_path}")

### Example documents with very low compression ratio

In [ ]:
cr_series = feature_df.set_index("doc_id")["compression_ratio"]

print("=" * 80)
print("TOP 5 LOWEST COMPRESSION RATIO DOCUMENTS")
print("=" * 80)
for doc_id in cr_series.nsmallest(5).index:
    score = cr_series[doc_id]
    snippet = str(text_series[doc_id])[:400].replace("\n", " ")
    print(f"\ndoc_id={doc_id}  compression_ratio={score:.4f}")
    print("-" * 60)
    print(snippet + " ...")

## Joint distribution: bigram TTR vs compression ratio

The two new features are plotted against each other.  Both axes point to the same type of anomaly — low lexical diversity — but they capture different aspects: bigram TTR measures phrase-level repetition while compression ratio measures byte-level redundancy across the full document.  Documents that score low on both simultaneously are prime anomaly candidates.

The scatter plot below is coloured by the Isolation Forest anomaly score computed on **all 17 structural features**, so we can see how well the two-dimensional projection aligns with the full-feature decision.

In [ ]:
# Fit Isolation Forest on all 17 structural features.
iso_model = IsolationForest(contamination=0.02, random_state=42)
iso_model.fit(feature_matrix)
anomaly_scores = -iso_model.score_samples(feature_matrix)  # higher = more anomalous
anomaly_mask = iso_model.predict(feature_matrix) == -1

feature_df["if_score"] = anomaly_scores
feature_df["is_anomaly"] = anomaly_mask

print(f"Isolation Forest flagged {anomaly_mask.sum()} documents as anomalies.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    feature_df["bigram_type_token_ratio"],
    feature_df["compression_ratio"],
    c=feature_df["if_score"],
    cmap="RdYlGn_r",
    s=8,
    alpha=0.7,
    linewidths=0,
)
plt.colorbar(scatter, ax=ax, label="Isolation Forest anomaly score (higher = more anomalous)")

# Highlight IF-flagged documents with a ring.
flagged = feature_df[feature_df["is_anomaly"]]
ax.scatter(
    flagged["bigram_type_token_ratio"],
    flagged["compression_ratio"],
    s=35,
    facecolors="none",
    edgecolors="black",
    linewidths=0.8,
    label=f"IF-flagged ({len(flagged)})",
    zorder=3,
)

ax.set_xlabel("Bigram type-token ratio", fontsize=11)
ax.set_ylabel("Compression ratio", fontsize=11)
ax.set_title("Bigram TTR vs Compression ratio\ncoloured by Isolation Forest anomaly score", fontsize=12)
ax.legend(fontsize=10, frameon=True)

plt.tight_layout()
scatter_path = RESULTS_DIR / "structural_bttr_vs_cr_scatter.png"
plt.savefig(scatter_path, bbox_inches="tight")
plt.show()
print(f"Saved → {scatter_path}")

**Reading the plot**: The two new features form a near-orthogonal signal.  Documents in the bottom-left corner have low phrase diversity *and* low information density — they compress heavily and repeat the same bigrams.  The IF-flagged documents (black rings) cluster tightly in that corner, confirming that structural redundancy is the primary signal the model uses.

## Feature correlation heatmap

A Pearson correlation heatmap reveals which features carry redundant information and which are independent signals.  Highly correlated feature pairs (|r| > 0.8) can be dropped without losing predictive power.

In [ ]:
corr_matrix = feature_df[feature_names].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.4,
    annot_kws={"size": 7},
    ax=ax,
)
ax.set_title("Pearson correlation between all 17 structural features", fontsize=12, pad=12)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)

plt.tight_layout()
heatmap_path = RESULTS_DIR / "structural_feature_correlation_heatmap.png"
plt.savefig(heatmap_path, bbox_inches="tight")
plt.show()
print(f"Saved → {heatmap_path}")

### Strong correlation pairs

The cell below identifies any feature pairs with |r| > 0.8, which would be candidates for removal in a leaner model.

In [ ]:
corr_pairs = (
    corr_matrix.where(np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["feature_a", "feature_b", "pearson_r"]
strong_pairs = corr_pairs[corr_pairs["pearson_r"].abs() > 0.8].sort_values("pearson_r", key=abs, ascending=False)

if strong_pairs.empty:
    print("No feature pairs with |r| > 0.8 — all features carry relatively independent information.")
else:
    print(f"Feature pairs with |Pearson r| > 0.8  ({len(strong_pairs)} pairs):")
    print(strong_pairs.to_string(index=False))

## Extreme documents per feature

For each of the 17 features, the document that scores at the most extreme (lowest) end is printed with a short text snippet.  This gives an intuitive sense of what drives each feature to its minimum.

In [ ]:
id_indexed_text = articles_df.set_index("doc_id")["text"]
id_indexed_features = feature_df.set_index("doc_id")

for feature_name in feature_names:
    extreme_doc_id = id_indexed_features[feature_name].idxmin()
    extreme_score = id_indexed_features.loc[extreme_doc_id, feature_name]
    snippet = str(id_indexed_text[extreme_doc_id])[:200].replace("\n", " ")
    print(f"\n{'─'*70}")
    print(f"Feature: {feature_name}   min={extreme_score:.5f}   doc_id={extreme_doc_id}")
    print(f"  {snippet} ...")

## Per-feature anomaly separation

The table below quantifies how well each feature separates the IF-flagged documents from the rest.  For each feature we report:

- **mean (flagged)** — average feature value among the IF-flagged documents.
- **mean (normal)** — average feature value among non-flagged documents.
- **ratio** — `mean(flagged) / mean(normal)`.  Values far from 1.0 indicate strong separation.

Features with the most extreme ratio are the ones driving the Isolation Forest decision.

In [ ]:
flagged_rows = feature_df[feature_df["is_anomaly"]][feature_names]
normal_rows = feature_df[~feature_df["is_anomaly"]][feature_names]

separation_df = pd.DataFrame({
    "mean_flagged": flagged_rows.mean(),
    "mean_normal": normal_rows.mean(),
})
separation_df["ratio"] = separation_df["mean_flagged"] / (separation_df["mean_normal"] + 1e-9)
separation_df = separation_df.sort_values("ratio")

pd.set_option("display.float_format", "{:.4f}".format)
print("Features sorted by flagged/normal ratio (lowest = most separating in the anomalous direction):")
separation_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(separation_df))
ax.bar(x - 0.2, separation_df["mean_normal"], 0.35, label="Normal documents", color="steelblue", alpha=0.85)
ax.bar(x + 0.2, separation_df["mean_flagged"], 0.35, label="IF-flagged documents", color="salmon", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(separation_df.index, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Mean feature value")
ax.set_title("Mean feature values: IF-flagged vs normal documents")
ax.legend(frameon=False)
plt.tight_layout()
bar_path = RESULTS_DIR / "structural_feature_separation_bar.png"
plt.savefig(bar_path, bbox_inches="tight")
plt.show()
print(f"Saved → {bar_path}")

## Export structural feature summary

Save a CSV with all 17 structural feature values and the Isolation Forest anomaly score for every document.  This file can be joined to the main anomalies or clusters output by `doc_id`.

In [ ]:
export_df = feature_df[["doc_id"] + feature_names + ["if_score", "is_anomaly"]].copy()
export_path = RESULTS_DIR / "structural_features_all_docs.csv"
export_df.to_csv(export_path, index=False)
print(f"Exported {len(export_df):,} rows × {export_df.shape[1]} columns → {export_path}")

## Summary

The 17 structural features provide a compact, parameter-free representation of document shape.  The separation table computed above shows the exact median values for both groups.  The two features with the most extreme flagged/normal ratios — **bigram type-token ratio** and **compression ratio** — are general measures of lexical and informational diversity.  Neither relies on any knowledge of what the anomalies look like; they were selected because low diversity is a theoretically motivated anomaly signal for any document corpus.

The Isolation Forest operating on these 17 structural features identifies a tight cluster of highly repetitive documents that are clearly separated from the main newsgroup distribution.  Operating on the full high-dimensional LSA embedding instead yields no such separation — the anomalous documents do not stand out semantically in that space.

### Files produced

| File | Description |
|---|---|
| `data/results/structural_features_overview.png` | 6×3 histogram grid for all 17 features |
| `data/results/structural_bttr_distribution.png` | Bigram TTR full and tail histograms |
| `data/results/structural_compression_ratio_distribution.png` | Compression ratio full and tail histograms |
| `data/results/structural_bttr_vs_cr_scatter.png` | 2D scatter coloured by IF anomaly score |
| `data/results/structural_feature_correlation_heatmap.png` | Pearson correlation heatmap |
| `data/results/structural_feature_separation_bar.png` | Mean feature values: flagged vs normal |
| `data/results/structural_features_all_docs.csv` | Feature matrix + IF scores for all documents |